In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Config variables
BATCH_SIZE = 128
EPOCHS = 5
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data pipeline
transform = transforms.Compose([
    transforms.ToTensor(), # [0,255] -> [0,1]
    transforms.Normalize((0.1307,),(0.3081,)) # MNIST mean/std
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [20]:
# Model
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(28 * 28, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        x = x.flatten(1)
        return self.net(x)

model = MLP().to(DEVICE)

# Loss + optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# Trainging loop
def train():
    model.train()
    for epoch in range (EPOCHS):
        total_loss = 0.0
        correct = 1
        total = 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            logits = model(x)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
            
        print(f"Epoch {epoch+1} | Loss: {total_loss/total:.4f} | Acc: {correct/total:.4f}")

        

In [21]:
# Evaluation
@torch.no_grad()
def evaluate():
    model.eval()
    correct = 0
    total = 0

    for x,y in test_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        preds = logits.argmax(dim=1)
        correct += (preds==y).sum().item()
        total += y.size(0)

    print(f"Test Accuracy: {correct/total:.4f}")


train()
evaluate()

Epoch 1 | Loss: 0.2714 | Acc: 0.9190
Epoch 2 | Loss: 0.1022 | Acc: 0.9685
Epoch 3 | Loss: 0.0693 | Acc: 0.9784
Epoch 4 | Loss: 0.0520 | Acc: 0.9834
Epoch 5 | Loss: 0.0386 | Acc: 0.9875
Test Accuracy: 0.9744
